In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
 !pip install -q faster-whisper ffmpeg-python opencv-python-headless yt-dlp rapidfuzz

In [ ]:
import os
import json
import time
import functools
import subprocess
import cv2
import torch
from faster_whisper import WhisperModel
from rapidfuzz import fuzz

In [ ]:
def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = round(time.perf_counter() - start, 3)
        if isinstance(result, dict):
            result["_stage_duration_sec"] = elapsed  # underscore-prefixed to avoid colliding
            # with any domain field a stage already returns (e.g. get_video_metadata's
            # own "duration_sec" for the video's length)
        return result
    return wrapper

In [ ]:
# %% [cell] 3. Config
VIDEO_URL_OR_PATH = "https://ok.ru/video/248244667877"   # <-- set this
TARGET_DIALOGUE   = "My mind rebels at stagnation"     # <-- set this
WORK_DIR          = "/kaggle/working"
VIDEO_PATH        = os.path.join(WORK_DIR, "input_video.mp4")
AUDIO_PATH        = os.path.join(WORK_DIR, "audio.wav")
FRAME_OUT_PATH    = os.path.join(WORK_DIR, "matched_frame.jpg")
 
MATCH_THRESHOLD   = 80.0   # rapidfuzz score (0-100) above which a fuzzy match counts as "partial_match"
COARSE_MATCH_THRESHOLD = 50.0
# --- Group A: tiering config ---
SHORT_MAX_SEC   = 180    # < 3 min speech  -> "short" tier
MEDIUM_MAX_SEC  = 1200   # < 20 min speech -> "medium" tier
                          # >= 20 min speech -> "long" tier
 
TIER_MODEL_MAP = {
    "short":  "large-v3",
    "medium": "large-v3",
    "long":   "medium"    # kept for reference; "long" tier now uses coarse-to-fine below instead
}
 
# --- Coarse-to-fine config (long tier only) ---
LONG_COARSE_MODEL = "small"       # fast first pass over the full (VAD-filtered) audio
LONG_FINE_MODEL = "large-v3"      # precise second pass, run only on the candidate window
CANDIDATE_WINDOW_BUFFER_SEC = 45  # seconds of padding on each side of the coarse match
AUDIO_SLICE_PATH = os.path.join(WORK_DIR, "audio_candidate_slice.wav")
 
# Model cache: load each faster-whisper model size once per Kaggle session, reuse after that
_MODEL_CACHE = {}
_VAD_CACHE = {}
def load_whisper_model(model_size: str) -> WhisperModel:
    if model_size not in _MODEL_CACHE:
        print(f"Loading faster-whisper model '{model_size}' (first use this session)...")
        _MODEL_CACHE[model_size] = WhisperModel(model_size, device=DEVICE, compute_type="float16")
    else:
        print(f"Reusing cached faster-whisper model '{model_size}'")
    return _MODEL_CACHE[model_size]
 
def load_vad_model():
    if "model" not in _VAD_CACHE:
        print("Loading Silero VAD model (first use this session)...")
        vad_model, utils = torch.hub.load(
            repo_or_dir="snakers4/silero-vad", model="silero_vad", trust_repo=True
        )
        _VAD_CACHE["model"] = vad_model
        _VAD_CACHE["utils"] = utils
    else:
        print("Reusing cached Silero VAD model")
    return _VAD_CACHE["model"], _VAD_CACHE["utils"]
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
 

In [ ]:
# STAGE 1: VIDEO ACQUISITION
# =============================================================================
# %% [cell] 4. Acquire video
@timed
def acquire_video(source: str, dest: str) -> dict:
    try:
        if source.startswith("http://") or source.startswith("https://"):
            cmd = ["yt-dlp", "-f", "mp4", "-o", dest, source]
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            path = dest
        else:
            if not os.path.exists(source):
                return {"status": "download_failed", "reason": f"Local file not found: {source}"}
            path = source
        return {"status": "ok", "path": path}
    except subprocess.CalledProcessError as e:
        return {"status": "download_failed", "reason": e.stderr or str(e)}
    except Exception as e:
        return {"status": "download_failed", "reason": str(e)}

In [ ]:

# STAGE 2: METADATA EXTRACTION
# =============================================================================
# %% [cell] 5. Extract fps, frame count, duration, VFR flag
@timed
def get_video_metadata(video_path: str) -> dict:
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return {"status": "metadata_failed", "reason": "Could not open video file"}
 
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps else 0
        cap.release()
 
        if not fps or fps <= 0:
            return {"status": "metadata_failed", "reason": "Invalid or unreadable fps"}
 
        probe = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=r_frame_rate,avg_frame_rate",
             "-of", "json", video_path],
            capture_output=True, text=True, check=True
        )
        probe_data = json.loads(probe.stdout)
        r_rate = probe_data["streams"][0]["r_frame_rate"]
        avg_rate = probe_data["streams"][0]["avg_frame_rate"]
        is_vfr = r_rate != avg_rate
 
        return {
            "status": "ok",
            "fps": fps,
            "total_frames": total_frames,
            "duration_sec": duration,
            "is_vfr": is_vfr
        }
    except Exception as e:
        return {"status": "metadata_failed", "reason": str(e)}

In [ ]:
# STAGE 3: AUDIO EXTRACTION
# =============================================================================
# %% [cell] 6. Check for audio stream, then extract mono 16kHz WAV
@timed
def extract_audio(video_path: str, audio_path: str) -> dict:
    try:
        probe = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a",
             "-show_entries", "stream=index", "-of", "json", video_path],
            capture_output=True, text=True, check=True
        )
        streams = json.loads(probe.stdout).get("streams", [])
        if not streams:
            return {"status": "no_audio_track", "reason": "No audio stream found in video"}
 
        cmd = ["ffmpeg", "-y", "-i", video_path, "-ar", "16000", "-ac", "1", "-vn", audio_path]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        return {"status": "ok", "path": audio_path}
    except subprocess.CalledProcessError as e:
        return {"status": "audio_extraction_failed", "reason": e.stderr or str(e)}
    except Exception as e:
        return {"status": "audio_extraction_failed", "reason": str(e)}
 
# %% [cell] 6a. Extract a short audio slice around a candidate timestamp (for the fine pass)
@timed
def extract_audio_slice(audio_path: str, start_sec: float, end_sec: float, out_path: str) -> dict:
    try:
        start_sec = max(0.0, start_sec)
        cmd = [
            "ffmpeg", "-y", "-i", audio_path,
            "-ss", str(start_sec), "-to", str(end_sec),
            "-ar", "16000", "-ac", "1", out_path
        ]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        return {"status": "ok", "path": out_path, "window_start_sec": start_sec}
    except subprocess.CalledProcessError as e:
        return {"status": "audio_extraction_failed", "reason": e.stderr or str(e)}
    except Exception as e:
        return {"status": "audio_extraction_failed", "reason": str(e)}
 
# =============================================================================
# STAGE 3b: VOICE ACTIVITY DETECTION (drives tiering + is reused as ASR's VAD filter)
# =============================================================================
# %% [cell] 6b. Run Silero VAD to get speech segments + total speech duration
@timed
def run_vad(audio_path: str) -> dict:
    try:
        vad_model, utils = load_vad_model()
        (get_speech_timestamps, _, read_audio, *_rest) = utils
 
        wav = read_audio(audio_path, sampling_rate=16000)
        speech_timestamps = get_speech_timestamps(wav, vad_model, sampling_rate=16000)
 
        if not speech_timestamps:
            return {"status": "no_audio_track", "reason": "VAD found no speech in audio"}
 
        total_speech_sec = sum(
            (seg["end"] - seg["start"]) / 16000 for seg in speech_timestamps
        )
 
        return {
            "status": "ok",
            "speech_segments": speech_timestamps,
            "total_speech_sec": round(total_speech_sec, 2)
        }
    except Exception as e:
        return {"status": "transcription_failed", "reason": f"VAD failed: {e}"}
 
# %% [cell] 6c. Classify tier from speech duration -> pick model size
def classify_tier(total_speech_sec: float) -> dict:
    if total_speech_sec < SHORT_MAX_SEC:
        tier = "short"
    elif total_speech_sec < MEDIUM_MAX_SEC:
        tier = "medium"
    else:
        tier = "long"
 
    return {"tier": tier, "model_size": TIER_MODEL_MAP[tier]}

In [ ]:

# STAGE 4: TRANSCRIPTION (VAD + ASR)
# =============================================================================
# %% [cell] 7. Transcribe with word-level timestamps
# speech_segments (from run_vad) are reused here via clip_timestamps, so VAD
# only runs once -- not once standalone (for tiering) and again inside faster-whisper.
@timed
def transcribe(audio_path: str, model_size: str = "large-v3", speech_segments=None) -> dict:
    try:
        model = load_whisper_model(model_size)

        # faster-whisper clip_timestamps expects a flat list [s1, e1, s2, e2, ...]
        # in seconds (floats).  Convert Silero's sample-index segments (16 kHz) to
        # that format.  When no VAD output is available we leave it None and let
        # faster-whisper's own internal VAD run instead (vad_filter=True below).
        clip_timestamps = None
        if speech_segments:
            clip_timestamps = []
            for seg in speech_segments:
                clip_timestamps.extend([seg["start"] / 16000, seg["end"] / 16000])

        segments_gen, info = model.transcribe(
            audio_path,
            word_timestamps=True,
            vad_filter=(clip_timestamps is None),  # only let faster-whisper VAD if we have no VAD output already
            clip_timestamps=clip_timestamps
        )

        # FIX: Consume the lazy generator with per-segment safety.
        # model.transcribe() returns a generator; errors inside it (e.g. None word
        # objects from malformed audio segments) only surface during iteration.
        # Wrapping each segment individually lets us skip corrupt entries instead of
        # crashing the entire transcription stage with 'NoneType is not subscriptable'.
        segments = []
        for seg in segments_gen:
            try:
                _ = seg.words  # materialise word list; surfaces any latent error here
                segments.append(seg)
            except Exception:
                continue  # skip irrecoverably corrupt segments

        if not segments:
            return {"status": "transcription_failed", "reason": "No speech detected in audio"}

        return {
            "status": "ok",
            "segments": segments,
            "language": info.language,
            "language_probability": info.language_probability
        }
    except Exception as e:
        return {"status": "transcription_failed", "reason": str(e)}


In [ ]:
# STAGE 5: DIALOGUE MATCHING (exact -> lexical fuzzy fallback)
# =============================================================================
# %% [cell] 8. Match dialogue: exact first, rapidfuzz fuzzy fallback otherwise
@timed
def match_dialogue(segments, target_text: str, threshold: float = MATCH_THRESHOLD) -> dict:
    target_norm = target_text.strip().lower()

    # --- Pass 1: exact substring match ---
    for seg in segments:
        seg_text_norm = (seg.text or "").strip().lower()

        if target_norm in seg_text_norm:
            target_words = target_norm.split()
            match_start_time = seg.start  # fallback: segment start

            # FIX 3: seg.words can be None even when seg.text exists (the model may
            # omit word-level data for some segments).  Guard explicitly.
            words = seg.words if seg.words is not None else []

            for i in range(len(words)):
                current_words = words[i:i + len(target_words)]

                # FIX 4: individual word objects inside the list can also be None
                # (faster-whisper bug with certain inputs).  Skip such windows.
                if not all(
                    w is not None and getattr(w, "word", None) is not None
                    for w in current_words
                ):
                    continue

                window = " ".join(
                    w.word.strip().lower()
                    for w in current_words
                )

                if window.startswith(target_words[0]):
                    if getattr(words[i], "start", None) is not None:
                        match_start_time = words[i].start
                    break

            return {
                "status": "success",
                "matched_text": (seg.text or "").strip(),
                "similarity_score": 100.0,
                "target_timestamp": match_start_time
            }

    # --- Pass 2: lexical fuzzy fallback across all segments ---
    best_score = -1.0
    best_seg = None

    for seg in segments:
        score = fuzz.token_sort_ratio(
            target_norm,
            (seg.text or "").strip().lower()
        )

        if score > best_score:
            best_score = score
            best_seg = seg

    if best_seg is None:
        return {
            "status": "no_match",
            "reason": "No segments available to compare"
        }

    result = {
        "matched_text": (best_seg.text or "").strip(),
        "similarity_score": best_score,
        "target_timestamp": best_seg.start
    }

    if best_score >= threshold:
        result["status"] = "partial_match"
    else:
        result["status"] = "no_match"

    return result


In [ ]:
# =============================================================================
# STAGE 6 + 7: FRAME LOCALIZATION + EXTRACTION
# =============================================================================
# %% [cell] 9. Convert timestamp -> frame number, extract frame
@timed
def extract_frame(video_path: str, timestamp_sec: float, fps: float, out_path: str) -> dict:
    try:
        frame_number = round(timestamp_sec * fps)
 
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return {"status": "frame_extraction_failed", "reason": "Could not reopen video file"}
 
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        success, frame = cap.read()
        cap.release()
 
        if not success or frame is None:
            return {"status": "frame_extraction_failed", "reason": f"Could not read frame {frame_number}"}
 
        cv2.imwrite(out_path, frame)
        return {"status": "ok", "frame_number": frame_number, "path": out_path}
    except Exception as e:
        return {"status": "frame_extraction_failed", "reason": str(e)}

In [ ]:
# MAIN PIPELINE: guard-clause chain, short-circuits on first failure
# =============================================================================
# %% [cell] 10. Run pipeline end to end
def run_pipeline(video_url_or_path: str, target_dialogue: str) -> dict:
    pipeline_start = time.perf_counter()
    timings = {}
 
    def record(stage_name: str, stage_result: dict):
        # _stage_duration_sec was injected by @timed; pull it into the shared timings dict
        timings[stage_name] = stage_result.pop("_stage_duration_sec", None)
 
    video_result = acquire_video(video_url_or_path, VIDEO_PATH)
    record("acquire_video", video_result)
    if video_result["status"] != "ok":
        return {**video_result, "timings": timings}
 
    video_path = video_result["path"]
 
    metadata_result = get_video_metadata(video_path)
    record("get_video_metadata", metadata_result)
    if metadata_result["status"] != "ok":
        return {**metadata_result, "timings": timings}
 
    audio_result = extract_audio(video_path, AUDIO_PATH)
    record("extract_audio", audio_result)
    if audio_result["status"] != "ok":
        return {**audio_result, "timings": timings}
 
    vad_result = run_vad(audio_result["path"])
    record("run_vad", vad_result)
    if vad_result["status"] != "ok":
        return {**vad_result, "timings": timings}
 
    tier_info = classify_tier(vad_result["total_speech_sec"])
 
    if tier_info["tier"] != "long":
        # --- short / medium: single-pass transcription, as before ---
        transcript_result = transcribe(
            audio_result["path"],
            model_size=tier_info["model_size"],
            speech_segments=vad_result["speech_segments"]
        )
        record("transcribe", transcript_result)
        if transcript_result["status"] != "ok":
            return {**transcript_result, "timings": timings}
 
        match_result = match_dialogue(transcript_result["segments"], target_dialogue)
        record("match_dialogue", match_result)
 
    else:
        # --- long: coarse pass (fast model, full audio) -> candidate window -> fine pass (large-v3, slice) ---
        coarse_result = transcribe(
            audio_result["path"],
            model_size=LONG_COARSE_MODEL,
            speech_segments=vad_result["speech_segments"]
        )
        record("transcribe_coarse", coarse_result)
        if coarse_result["status"] != "ok":
            return {**coarse_result, "timings": timings}
 
        coarse_match = match_dialogue(coarse_result["segments"], target_dialogue, threshold=COARSE_MATCH_THRESHOLD)
        record("match_dialogue_coarse", coarse_match)
        if coarse_match["status"] not in ("success", "partial_match"):
            # coarse pass couldn't even roughly locate it -- no point running the expensive fine pass
            timings["total"] = round(time.perf_counter() - pipeline_start, 3)
            return {
                "status": coarse_match["status"],
                "query": target_dialogue,
                "closest_text": coarse_match.get("matched_text"),
                "similarity_score": coarse_match.get("similarity_score"),
                "reason": coarse_match.get("reason", "Coarse pass found no usable candidate"),
                "tier_info": tier_info,
                "timings": timings
            }
 
        candidate_time = coarse_match["target_timestamp"]
        window_start = max(0.0, candidate_time - CANDIDATE_WINDOW_BUFFER_SEC)
        window_end = min(metadata_result["duration_sec"], candidate_time + CANDIDATE_WINDOW_BUFFER_SEC)
 
        slice_result = extract_audio_slice(audio_result["path"], window_start, window_end, AUDIO_SLICE_PATH)
        record("extract_audio_slice", slice_result)
        if slice_result["status"] != "ok":
            return {**slice_result, "timings": timings}
 
        fine_result = transcribe(slice_result["path"], model_size=LONG_FINE_MODEL, speech_segments=None)
        record("transcribe_fine", fine_result)

        # FIX 5: If the fine-pass transcription fails (e.g. None word objects on a
        # short/noisy slice), do NOT abort the entire pipeline.  Fall back gracefully
        # to the coarse match result, flagged as lower-confidence.  Previously this
        # returned {status: transcription_failed} and the frame was never extracted.
        if fine_result["status"] != "ok":
            print(f"[WARN] Fine-pass transcription failed ({fine_result.get('reason')}); "
                  "falling back to coarse-pass result.")
            match_result = coarse_match
            match_result["status"] = "partial_match"
            match_result["note"] = (
                f"Fine-pass transcription failed: {fine_result.get('reason')}. "
                "Using coarse-pass timestamp (lower confidence)."
            )
        else:
            fine_match = match_dialogue(fine_result["segments"], target_dialogue)
            record("match_dialogue_fine", fine_match)

            if fine_match["status"] not in ("success", "partial_match"):
                # fine pass couldn't confirm within the window -- fall back to the coarse estimate,
                # explicitly flagged as lower-confidence rather than silently trusted as fine-verified
                match_result = coarse_match
                match_result["status"] = "partial_match"
                match_result["note"] = "Fine-pass verification failed; using coarse-pass estimate (lower confidence)."
            else:
                # offset the slice-local timestamp back into the full video's timeline
                fine_match["target_timestamp"] += slice_result["window_start_sec"]
                match_result = fine_match
 
    if match_result["status"] not in ("success", "partial_match"):
        timings["total"] = round(time.perf_counter() - pipeline_start, 3)
        # no_match: still include the closest candidate info for debugging, but don't extract a frame
        return {
            "status": match_result["status"],
            "query": target_dialogue,
            "closest_text": match_result.get("matched_text"),
            "similarity_score": match_result.get("similarity_score"),
            "reason": match_result.get("reason"),
            "timings": timings
        }
 
    frame_result = extract_frame(
        video_path, match_result["target_timestamp"], metadata_result["fps"], FRAME_OUT_PATH
    )
    record("extract_frame", frame_result)
    if frame_result["status"] != "ok":
        return {**frame_result, "timings": timings}
 
    timings["total"] = round(time.perf_counter() - pipeline_start, 3)
 
    return {
        "status": match_result["status"],  # "success" or "partial_match"
        "query": target_dialogue,
        "matched_text": match_result["matched_text"],
        "similarity_score": match_result["similarity_score"],
        "timestamp_sec": round(match_result["target_timestamp"], 3),
        "note": match_result.get("note"),  # present only on the coarse-fallback path
        "frame_number": frame_result["frame_number"],
        "frame_image_path": frame_result["path"],
        "video_metadata": {
            "fps": metadata_result["fps"],
            "duration_sec": metadata_result["duration_sec"],
            "is_vfr": metadata_result["is_vfr"]
        },
        "tier_info": {
            "tier": tier_info["tier"],
            "model_size": tier_info["model_size"],
            "total_speech_sec": vad_result["total_speech_sec"]
        },
        "timings": timings
    }


In [ ]:
# %% [cell] 11. Run it
result = run_pipeline(VIDEO_URL_OR_PATH, TARGET_DIALOGUE)
print(json.dumps(result, indent=2))
 
# %% [cell] 12. Display frame if extraction succeeded
if result["status"] in ("success", "partial_match") and result.get("frame_image_path"):
    from IPython.display import Image, display
    display(Image(filename=result["frame_image_path"]))
